# Project assignment 2

In [11]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds

IMG_SIZE = (64, 64)
BATCH_SIZE = 32

# train_data = tf.keras.utils.image_data_set_from_directory(
#     'dataset/train',
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
#     label_mode='categorical'
# )

# val_data = tf.keras.utils.image_data_set_from_directory(
#     'dataset/validation',
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
#     label_mode='categorical'
# )

# val_data = tf.keras.utils.image_data_set_from_directory(
#     'dataset/test',
#     target_size=IMG_SIZE,
#     batch_size=BATCH_SIZE
#     label_mode='categorical'
# )

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

TEST

In [12]:
# 1. Ladataan kissa/koira-data netistä ja jaetaan se suhteessa 70/15/15
(train_raw, val_raw, test_raw), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    with_info=True,
    as_supervised=True,
)

# 2. Esikäsittelyfunktio (Koon muutos 224x224 ja normalisointi)
def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))
    image = image / 255.0  # Skaalaus 0-1 välille
    label = tf.one_hot(label, 2) # Muutetaan categorical-muotoon (2 luokkaa)
    return image, label

# 3. Luodaan lopulliset datasetit
BATCH_SIZE = 32

train_data = train_raw.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_data = val_raw.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_data = test_raw.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Data ladattu! Luokkia: {metadata.features['label'].num_classes}")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.BQNOFP_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Data ladattu! Luokkia: 2


Model 1 (CNN)

In [13]:
model1 = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax')
])

Model 2 (VGG16)

In [14]:
base_model = tf.keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model2 = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(2, activation='softmax')
])

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model 3 (Fine tuning)

In [15]:
base_model = keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model2 = keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(2, activation='softmax')
])
model2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])